<a href="https://colab.research.google.com/github/darlim9141/kcu5/blob/main/cnn-notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Study Note] Fashion Style Classification using ResNet-50

## 1. Introduction

### 1.1. Objective
Following the validation of our dataset through unsupervised learning, we now proceed to **Supervised Learning**. The goal is to train a Deep Neural Network to classify a given fashion image into one of four categories: *Minimal, Casual, Classic, or Street*.

### 1.2. The Challenge: Small Data
Deep Learning typically requires massive datasets (thousands of images per class) to learn effective features from scratch. However, our dataset is relatively small (~500 images). To overcome this limitation and achieve high accuracy, we employ two key strategies:
1.  **Data Augmentation:** Artificially expanding the dataset.
2.  **Transfer Learning:** Leveraging knowledge from a pre-trained model.

## 2. Data Preprocessing & Augmentation

### 2.1. Preprocessing
Neural networks operate on numerical tensors.
* **Resizing:** All images are resized to $(224 \times 224 \times 3)$ to match the input requirement of the ResNet model.
* **Normalization:** Pixel intensity values (0-255) are scaled to the range $[0, 1]$. This ensures faster convergence during gradient descent.

### 2.2. Data Augmentation
**Overfitting** occurs when a model memorizes the training data but fails to generalize to new data. Augmentation mitigates this by applying random transformations to training images, forcing the model to learn invariant features (e.g., recognizing a shirt regardless of its angle).

* **Rotation & Shift:** Simulates different camera angles.
* **Horizontal Flip:** Simulates mirror images.
* **Zoom/Shear:** Simulates distance and perspective changes.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Configuration
# Update PATH to your Google Drive or local path
BASE_DIR = './dataset'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4

# 2. Data Generators (Preprocessing + Augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,         # Normalization
    rotation_range=20,      # Rotate image by up to 20 degrees
    width_shift_range=0.2,  # Shift width by 20%
    height_shift_range=0.2, # Shift height by 20%
    shear_range=0.2,        # Shear transformation
    zoom_range=0.2,         # Zoom in/out
    horizontal_flip=True,   # Flip horizontally
    fill_mode='nearest',    # Fill missing pixels
    validation_split=0.2    # 20% reserved for validation
)

# Note: Validation data should NOT be augmented (only rescaled)
# In this implementation, we use the same generator for simplicity but subsetting handles the split.

print("Data Generators set up complete.")
# In a real scenario, you would uncomment lines below to load data:
# train_generator = train_datagen.flow_from_directory(BASE_DIR, target_size=IMG_SIZE, subset='training', ...)
# validation_generator = train_datagen.flow_from_directory(BASE_DIR, target_size=IMG_SIZE, subset='validation', ...)

## 3. Model Architecture: Convolutional Neural Networks (CNN)

### 3.1. What is a CNN?
A **Convolutional Neural Network (CNN)** is a deep learning architecture designed for processing grid-like data, such as images.
* **Convolutional Layer:** Uses learnable filters (kernels) to scan the image and extract local features (edges, textures).
* **Pooling Layer:** Reduces the spatial dimensions (downsampling) to decrease computation and extract dominant features.

### 3.2. Why ResNet-50? (Residual Networks)
As neural networks become deeper, they suffer from the **Vanishing Gradient Problem**, where gradients become too small for the network to learn effectively during backpropagation.

**ResNet (Residual Network)** solves this using **Skip Connections (Residual Blocks)**.
* Instead of learning the mapping $H(x)$ directly, it learns the residual function $F(x) = H(x) - x$.
* The skip connection allows the gradient to flow directly through the network, enabling the training of very deep networks (e.g., 50 layers, 101 layers).

### 3.3. Transfer Learning Strategy
We utilize **Fine-Tuning**:
1.  **Base Model:** Load ResNet-50 pre-trained on **ImageNet** (1.2M images). We freeze these layers to preserve the learned feature extractors.
2.  **Custom Head:** Add new layers on top to classify our specific 4 fashion styles.
    * **GlobalAveragePooling2D:** Converts 3D feature maps into a 1D vector.
    * **Dropout:** Randomly deactivates neurons during training to prevent overfitting.
    * **Softmax:** Outputs probability distribution across the 4 classes.

In [ ]:
def build_model(num_classes=4):
    # 1. Load Pre-trained Base Model
    # include_top=False: Exclude the original 1000-class classification layer
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    # Freeze the base model to prevent destroying learned features during initial training
    base_model.trainable = False

    # 2. Construct Custom Classification Head
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(), # Feature Map (7x7x2048) -> Vector (2048)
        layers.Dense(512, activation='relu'), # Intermediate dense layer
        layers.Dropout(0.5), # Regularization
        layers.Dense(num_classes, activation='softmax') # Output layer
    ])

    return model

model = build_model()
model.summary()

## 4. Training Configuration

### 4.1. Loss Function: Categorical Crossentropy
Since we are performing multi-class classification (4 classes), we use **Categorical Crossentropy**. It measures the difference between the true probability distribution (one-hot encoded) and the predicted probability distribution.

### 4.2. Optimizer: Adam
**Adam (Adaptive Moment Estimation)** is an optimization algorithm that adapts the learning rate for each parameter. It combines the advantages of AdaGrad and RMSProp, making it robust and fast for most deep learning tasks.

In [ ]:
# Compile the model
model.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled ready for training.")

# --- Training Simulation Code (Placeholder) ---
# history = model.fit(
#     train_generator,
#     validation_data=validation_generator,
#     epochs=EPOCHS,
#     callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)]
# )

## 5. Qualitative Analysis: Explainable AI (XAI)

### 5.1. The "Black Box" Problem
Deep learning models are often criticized for being "black boxes" because it is difficult to understand *why* they made a specific decision.

### 5.2. Grad-CAM (Gradient-weighted Class Activation Mapping)
To address this, we use **Grad-CAM**.
* **Mechanism:** It uses the gradients of the target concept flowing into the final convolutional layer to produce a coarse localization map.
* **Result:** It generates a **Heatmap** overlaid on the original image, highlighting the regions the model focused on (e.g., highlighting the collar and buttons when classifying a "Shirt").

*(Note: The implementation of Grad-CAM requires accessing internal model layers and gradients, which is performed in the evaluation phase.)*